# 실습 C. 변경 요청 1 "대기열 늘리기" 판단하기 (약 40분)

**상황.** 상품기획팀 요청: 저녁 시간에 이벤트가 몰리면 허브가 이벤트를 거부합니다. 최대 32개까지 받게 해 주세요. 단 메모리가 작으니 평소에는 8칸만 쓰고 필요할 때만 늘려 주세요. 규칙 R1~R4 중 **R1의 상한 값(8 → 32)만** 바꾸는 것이 승인됐습니다.

AI에게 이 요청을 줬더니 패치 두 개가 나왔습니다. **둘 다 테스트를 전부 통과합니다.** 무엇을 내보낼 수 있을까요?

| | 하는 일 |
|---|---|
| 패치 A | 대기열이 가득 차면 배열을 2배로 늘림(최대 32). `grow()`가 배열을 **통째로 복사**하고 `head`는 그대로 |
| 패치 B | 늘릴 때 **오래된 것부터 순서대로** 복사하고 `head = 0`. 그리고 상한 32를 Dispatcher와 Watchdog이 설정값 `dispatch.maxPending`으로 직접 가짐 |

In [1]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## 무엇이 바뀌었나

### 패치 A

In [2]:
lab.run("show", V="scenarios/req1/ai-A")

$ make show V=scenarios/req1/ai-A


**생성자가 두 개가 됐습니다.** 인자 2개짜리 생성자 `(시작 칸 수, 최대 칸 수)`가 새로 생겼고, 원래 있던 인자 1개짜리는 "시작은 그 칸 수, 최대 32칸"이 됐습니다.

In [3]:
lab.show("scenarios/req1/ai-A/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 13, 35, mark=[14, 26, 29], title="패치 A의 RingEventQueue 생성자")

| 생성자 | 레거시 | 패치 A |
|---|---|---|
| `new RingEventQueue(8)` (인자 1개) | 8칸, 늘어나지 않음 | 8칸에서 시작, **32칸까지 늘어남** (26줄) |
| `new RingEventQueue(3, 3)` (인자 2개, 새로 생김) | 없음 | 3칸에서 시작, 최대 3칸 = 늘어나지 않음 |

**테스트를 `(3)` → `(3, 3)`으로 고친 이유.** `DispatcherTest`는 빨리 가득 차게 하려고 3칸 대기열을 써서 "가득 차면 R2~R4대로 거부·밀어내기"를 확인합니다. 패치 후 `new RingEventQueue(3)`은 32칸까지 늘어나 가득 차지 않으므로, 고치지 않으면 이 테스트들이 실패합니다. 직접 봅니다(`ai-A-oldtests`: 패치 A의 제품 코드 + 레거시 테스트 그대로).

In [4]:
lab.run("variant-test", V="scenarios/req1/ai-A-oldtests")

$ make variant-test V=scenarios/req1/ai-A-oldtests


tests: 38, passed: 33, failed: 5


"가득 찼을 때" 테스트 5개가 실패합니다. AI는 대기열을 `(3, 3)`, 즉 늘어나지 않는 3칸으로 바꿨습니다. 가득 찼을 때의 규칙 R2~R4는 바뀌지 않았으니, **이 수정 자체는 합리적입니다.**

In [5]:
lab.diff("legacy/src/test/java/com/dxlab/homehub/DispatcherTest.java", "scenarios/req1/ai-A/src/test/java/com/dxlab/homehub/DispatcherTest.java", title="AI가 고친 DispatcherTest (패치 A)")

**문제는 새 스펙을 확인하는 테스트가 없다는 것입니다.** 패치 A가 더한 테스트 2개는 빈 대기열에서 늘리기만 합니다. 그래서 다음 경우는 어떤 테스트도 만들지 않습니다.

- 대기열이 한 바퀴 돈 상태(head가 0이 아님)에서 늘어나는 경우
- Dispatcher·Watchdog이 늘어나는 대기열을 쓰는 경우(실제 허브는 인자 1개짜리 생성자로 늘어나는 대기열을 씀)

In [6]:
lab.show("legacy/src/main/java/com/dxlab/homehub/app/HomeHub.java", 43, 46, mark=[44], title="HomeHub: 인자 1개짜리 생성자 → 패치 A에서는 32칸까지 늘어나는 대기열")

### 패치 B

대기열은 순서대로 복사하고, 상한은 Dispatcher와 Watchdog이 직접 갖습니다. 테스트도 `(3, 3)`으로 고친 것은 같지만, **새 스펙을 확인하는 테스트를 더했습니다**: 늘어나는 대기열에서 Dispatcher가 상한을 지키는지(`testLogicalLimitAboveInitialCapacity`), 허브 전체에 이벤트 32개가 들어가고 33번째가 거부되는지(`testBurstOf32IsAccepted`).

In [7]:
lab.run("show", V="scenarios/req1/ai-B")

$ make show V=scenarios/req1/ai-B


## C-1. 테스트만으로 판단하기 (5분)

In [8]:
lab.run("req1-tests")

$ make req1-tests


ai-A   tests: 40, passed: 40, failed: 0


ai-B   tests: 42, passed: 42, failed: 0


패치 A 40/40, 패치 B 42/42. 테스트만 보면 둘 다 내보낼 수 있습니다.

**생각해 볼 것.** 테스트 코드의 변경을 리뷰할 때 확인할 것은 두 가지입니다. (1) 기존 테스트를 고친 것이 타당한가(`(3, 3)`은 타당), (2) **바뀐 스펙을 확인하는 테스트가 있는가**. 패치 A와 B는 (2)에서 어떻게 다릅니까?

## C-2. Dafny로 판단하기 (15분)

각 패치를 그대로 옮긴 Dafny 모델을 검증합니다. 계약(R1~R4)은 요청서가 승인한 상한 변경만 반영했습니다. 보고서에는 증명에 실패한 곳이 Dafny 코드와 함께(빨간 줄: 오류 위치, 노란 줄: 관련된 계약) 우리말 설명을 붙여 나옵니다.

In [9]:
lab.run("req1-verify")

$ make req1-verify


Dafny Types.dfy RingQueue_A.dfy                검증 24개, 오류 1개


Dafny Types.dfy RingQueue.dfy Dispatcher.dfy P 검증 73개, 오류 0개


**패치 A: `Grow`에서 오류.** Grow가 끝난 뒤 "배열을 head부터 읽으면 대기열 내용과 같다"(`Valid`)는 약속을 지키지 못합니다. 대기열이 한 바퀴 돈 상태에서 늘리면 순서가 깨집니다.

```
늘리기 전  (head=2, 내용 c,d,e,f)
  [ e | f | c | d ]                       c,d가 뒤쪽, e,f가 앞쪽에 있음

패치 A    (통째로 복사, head=2 유지)
  [ e | f | c | d | _ | _ | _ | _ ]       head=2부터 읽으면 c,d,_,_ ...

패치 B    (순서대로 복사, head=0)
  [ c | d | e | f | _ | _ | _ | _ ]
```

패치 A가 더한 테스트는 새 대기열(head=0)에서만 늘려 보기 때문에 이 상황을 만들지 않습니다.

**패치 B: 대기열과 Dispatcher 모두 통과.** Dispatcher 모델의 R1~R4 문구는 그대로이고, 상한을 대기열 크기가 아니라 승인된 값에서 가져오도록 바뀌었습니다.

In [10]:
lab.diff("formal/Dispatcher.dfy", "scenarios/req1/model/Dispatcher.dfy", title="Dispatcher 모델: 기준 모델 → 패치 B")

## C-3. 차분 테스트로 판단하기 (10분)

Dafny는 Java를 **옮긴 모델**을 검증합니다. 그래서 같은 연산을 Java와 패치 B의 모델(위에서 검증된 것)에 넣어 실제 Java도 같은지 봅니다.

| 비교 | 예상 | 뜻 |
|---|---|---|
| 패치 A vs 모델 | 불일치 | 실제 Java에서도 위의 버그가 남 |
| 패치 B vs 모델 | 일치 | |
| 레거시 vs 모델 | 불일치 | 승인된 동작 변화(상한 증가) |

In [11]:
lab.run("req1-difftest")

$ make req1-difftest


[queue] MISMATCH (run 0, seed 42)


[queue] 300 runs, 18000 ops: Java and model agree


[dispatcher] 300 runs, 18300 ops: Java and model agree


[power] 300 runs, 18000 ops: Java and model agree


[queue] MISMATCH (run 0, seed 42)


- **패치 A**: 연산 11개 뒤 마지막 단계에서 이벤트 14가 사라지고 빈 칸(null)이 생겼습니다. 이 연산열을 `RingEventQueueTest`에 회귀 테스트로 옮기면 패치 A의 테스트가 실패합니다.
- **레거시**: 5번째 `offer`에서 Java는 `FULL`, 모델은 `OK`(차분 테스트는 대기열을 4칸으로 만듭니다). 이것은 버그가 아니라 **이번 요청으로 승인된 변화**입니다. 차분 테스트는 "다르다"까지만 알려 줍니다. 그 차이가 승인된 것인지는 사람이 요청서와 맞춰 봐야 합니다.

## 정리

| | 테스트 | Dafny | 차분 테스트 | 판단 |
|---|---|---|---|---|
| 패치 A | 40/40 통과 | 대기열 `Grow` 실패 | 불일치 (이벤트가 사라짐) | 버그. 내보낼 수 없음 |
| 패치 B | 42/42 통과 | 통과 | 일치 | 내보낼 수 있음 |

패치 B에서도 확인할 것 하나: Watchdog의 경보 기준이 "8칸의 75%(6개)"에서 "승인된 상한 32의 75%(24개)"로 바뀝니다. 이것이 요청서가 승인한 범위인지는 사람이 확인합니다(A-2 ①에서 본 것과 같은 종류의 변화).

모든 결과를 한 장에 모은 보고서도 만들 수 있습니다.

In [12]:
lab.run("req1")

$ make req1


ai-A   tests: 40, passed: 40, failed: 0


ai-B   tests: 42, passed: 42, failed: 0


Dafny Types.dfy RingQueue_A.dfy                검증 24개, 오류 1개


Dafny Types.dfy RingQueue.dfy Dispatcher.dfy P 검증 73개, 오류 0개


[queue] MISMATCH (run 0, seed 42)


[queue] 300 runs, 18000 ops: Java and model agree


[dispatcher] 300 runs, 18300 ops: Java and model agree


[power] 300 runs, 18000 ops: Java and model agree


[queue] MISMATCH (run 0, seed 42)
